In [2]:
import sqlite3
conn = sqlite3.connect("../data/checkpoints.db")
# 查看有哪些线程（对话）存档
print(conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())

[('checkpoints',), ('writes',)]


In [3]:
# ============================================================
# 方式一（推荐）：用 LangGraph 的 SqliteSaver 读取完整对话
# 能正确反序列化消息对象（human / ai / tool），还原对话内容
# 注意：本 notebook 的 Kernel 需使用 langgraph 环境
# ============================================================
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("../data/checkpoints.db", check_same_thread=False)
saver = SqliteSaver(conn)

# 1. 列出所有对话线程
threads = [r[0] for r in conn.execute("SELECT DISTINCT thread_id FROM checkpoints")]
print(f"数据库中共 {len(threads)} 个对话线程: {threads}\n")

# 2. 读取每个线程的完整对话（最新检查点 = 完整快照）
for tid in threads:
    latest = None
    for cfg, cp, meta, parent, pending in saver.list({"configurable": {"thread_id": tid}}):
        latest = cp       # list() 从新到旧，第一条即最新完整快照
        break
    if latest is None:
        continue
    msgs = latest["channel_values"]["messages"]
    print(f"========== thread: {tid} （{len(msgs)} 条消息） ==========")
    for m in msgs:
        print(f"  [{m.type:<12}] {m.content}")
    print()

数据库中共 2 个对话线程: ['test-001', 'user-8a560764']

========== thread: test-001 （2 条消息） ==========
  [human       ] 你好
  [ai          ] 你好呀！我是小智，你的智能家居管家~ 🏠✨

有什么我可以帮你的吗？比如控制灯光、空调、电视，或者帮你设置一个舒适的场景模式？

========== thread: user-8a560764 （10 条消息） ==========
  [human       ] 你好
  [ai          ] 你好呀！我是小智，你的智能家居管家~ 🏠✨

有什么我可以帮你的吗？比如：
- 控制家里的灯光、空调、电视或窗帘
- 查看当前设备状态
- 激活智能场景（回家、离家、睡眠、观影、起床模式）

随时告诉我你的需求哦！😊
  [human       ] 现在家里什么状况
  [ai          ] 
  [tool        ] 🏠 **当前所有设备状态:**

**💡 灯光**
  · 客厅灯 (living_room_light): 🔴 关闭 | 亮度: 80% | 色温: 暖白
  · 卧室灯 (bedroom_light): 🔴 关闭 | 亮度: 60% | 色温: 暖白
  · 厨房灯 (kitchen_light): 🔴 关闭 | 亮度: 100% | 色温: 白光
**❄️ 空调**
  · 客厅空调 (living_room_ac): 🔴 关闭 | 温度: 26°C | 模式: 制冷 | 风速: 自动
  · 卧室空调 (bedroom_ac): 🔴 关闭 | 温度: 26°C | 模式: 制冷 | 风速: 自动
**📺 电视**
  · 客厅电视 (living_room_tv): 🔴 关闭 | 音量: 30%  | 输入源: HDMI 1
**🪟 窗帘**
  · 客厅窗帘 (living_room_curtain): 完全关闭
  · 卧室窗帘 (bedroom_curtain): 完全关闭
  [ai          ] 给你汇报一下家里的情况~ 🏠

**💡 灯光**
- 客厅灯、卧室灯、厨房灯 → 全部关闭

**❄️ 空调**
- 客厅空调、卧室空调 → 全部关闭（设

In [4]:
# ============================================================
# 方式二：直接查看原始 SQL 表结构
# 注意：checkpoint / value 字段是 msgpack 序列化（type 列标注），
#       直接用 json.loads 会失败，必须走 SqliteSaver 反序列化
# ============================================================
import sqlite3
conn = sqlite3.connect("../data/checkpoints.db")

for table in ("checkpoints", "writes"):
    print(f"========== {table} 表 ==========")
    for row in conn.execute(f"PRAGMA table_info({table})"):
        print(f"  {row[1]:<22} {row[2]}")
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  共 {n} 行\n")

# 看字段序列化格式
print("checkpoints.type 分布:", conn.execute(
    "SELECT type, COUNT(*) FROM checkpoints GROUP BY type").fetchall())

========== checkpoints 表 ==========
  thread_id              TEXT
  checkpoint_ns          TEXT
  checkpoint_id          TEXT
  parent_checkpoint_id   TEXT
  type                   TEXT
  checkpoint             BLOB
  metadata               BLOB
  共 16 行

========== writes 表 ==========
  thread_id              TEXT
  checkpoint_ns          TEXT
  checkpoint_id          TEXT
  task_id                TEXT
  idx                    INTEGER
  channel                TEXT
  type                   TEXT
  value                  BLOB
  共 20 行

checkpoints.type 分布: [('msgpack', 16)]
